# 🚀 Entrenamiento LSTM - MEDIUM V3 (120→7 días)

**GPU disponible:** Tesla T4 (gratis)
**Tiempo estimado:** 1-2 horas
**Costo:** $0

---

## ⚙️ IMPORTANTE: Activar GPU
1. **Runtime** → **Change runtime type**
2. **Hardware accelerator**: **GPU** (T4)
3. Click **Save**

---

## 1️⃣ Verificar GPU

In [ ]:
import tensorflow as tf
import gc

gc.collect()

print("="*80)
print("CONFIGURACIÓN DE COLAB - MEDIUM V3")
print("="*80)
print(f"TensorFlow version: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"GPU disponible: {gpus}")

if len(gpus) > 0:
    print(f"\n✅ {len(gpus)} GPU(s) DETECTADA(S)")
    for i, gpu in enumerate(gpus):
        print(f"   GPU {i}: {gpu.name}")
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as e:
            print(f"   Advertencia: {e}")
else:
    print("\n⚠️ GPU NO DETECTADA - Ve a Runtime → Change runtime type → GPU")

print("="*80)

## 2️⃣ Instalar dependencias

In [ ]:
!pip install -q openpyxl seaborn
print("\n✅ Dependencias instaladas")

import gc
gc.collect()

## 3️⃣ Subir archivos

Sube estos 2 archivos cuando se te solicite:
- `train_all_customers_temporal_3.py`
- `online_retail_2.xlsx`

In [ ]:
from google.colab import files
import os
import gc

print("📤 Sube 'train_all_customers_temporal_3.py'")
uploaded = files.upload()

print("\n📤 Sube 'online_retail_2.xlsx'")
uploaded = files.upload()

# Crear estructura
!mkdir -p data/processed
!mkdir -p models/temporal/customer_v3/medium

# Mover archivos
!mv online_retail_2.xlsx data/processed/

print("\n✅ Archivos subidos y organizados")
!ls -lh data/processed/
!ls -lh *.py

gc.collect()

## 4️⃣ Importar script de entrenamiento

In [ ]:
import sys
import gc

sys.path.append('.')

try:
    from train_all_customers_temporal_3 import CustomerTemporalAnalyzer, TemporalConfig
    
    # ⚠️ OPTIMIZACIÓN AGRESIVA DE MEMORIA PARA COLAB
    print("⚙️ Aplicando optimizaciones agresivas de memoria...")
    TemporalConfig.MEDIUM['batch_size'] = 16  # Reducido de 64 a 16 (como LONG en Kaggle)
    
    print("✅ Script V3 importado correctamente")
    print(f"\n📊 Configuración MEDIUM V3 (optimizada para Colab):")
    print(f"   Window: {TemporalConfig.MEDIUM['window_days']} días")
    print(f"   Forecast: {TemporalConfig.MEDIUM['forecast_days']} días")
    print(f"   Epochs: {TemporalConfig.MEDIUM['epochs']}")
    print(f"   Batch size: {TemporalConfig.MEDIUM['batch_size']} (bajo uso de RAM)")
except Exception as e:
    print(f"❌ ERROR: {e}")
    raise

gc.collect()

## 5️⃣ Preparar datos (con optimización de memoria)

In [ ]:
import warnings
import gc
import numpy as np
warnings.filterwarnings('ignore')
from datetime import datetime

gc.collect()

start_time = datetime.now()
print(f"⏰ Inicio: {start_time.strftime('%Y-%m-%d %H:%M:%S')}\n")

print("="*70)
print("INICIALIZANDO ANALYZER V3")
print("="*70)

analyzer = CustomerTemporalAnalyzer(
    data_path='data/processed/online_retail_2.xlsx',
    output_dir='models/temporal/customer_v3'
)

print("\n" + "="*70)
print("FASE 1: Preparación de datos")
print("="*70)

print("\n🔄 Paso 1/3: Cargando datos...")
analyzer.load_and_preprocess_data()
gc.collect()

print("\n🔄 Paso 2/3: Calculando RFM...")
analyzer.calculate_rfm_metrics()
gc.collect()

print("\n🔄 Paso 3/3: Generando secuencias...")
analyzer.generate_customer_sequences(min_transactions=5)

# ⚠️ OPTIMIZACIÓN CRÍTICA: Reducir clientes para evitar OOM
print("\n⚙️ OPTIMIZACIÓN AGRESIVA DE MEMORIA:")
print(f"   Clientes totales: {len(analyzer.customers)}")

# Seleccionar top 1000 clientes (igual que Kaggle LONG)
analyzer.customers = sorted(analyzer.customers, 
                           key=lambda x: x['TotalPurchases'], 
                           reverse=True)[:1000]

print(f"   Clientes seleccionados: {len(analyzer.customers)} (top 1000 por compras)")
print(f"   Promedio compras/cliente: {np.mean([c['TotalPurchases'] for c in analyzer.customers]):.1f}")

gc.collect()

print("\n" + "="*70)
print("✅ DATOS PREPARADOS - RAM optimizada al máximo")
print("="*70)

## 6️⃣ ENTRENAR MEDIUM V3

**Esta celda tomará 1-2 horas.**

In [ ]:
import time
import gc

print(f"\n{'='*70}")
print(f"FASE 2: Entrenamiento MEDIUM V3 (120→7 días)")
print(f"{'='*70}")

horizon_start = time.time()

try:
    gc.collect()
    
    print("\n🚀 Iniciando entrenamiento (30 epochs)...\n")
    model, history, metrics = analyzer.train_horizon_model(TemporalConfig.MEDIUM)
    
    duration = (time.time() - horizon_start) / 60
    
    print(f"\n✅ ENTRENAMIENTO COMPLETADO en {duration:.1f} minutos")
    print("="*70)
    print(f"   Accuracy: {metrics['purchase_prob_accuracy']*100:.2f}%")
    print(f"   AUC: {metrics['purchase_prob_auc']:.4f}")
    print(f"   Days MAE: {metrics['days_mae']:.2f}")
    print(f"   Value MAE: ${metrics['value_mae']:.2f}")
    print("="*70)
    
    del model
    del history
    gc.collect()
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()

end_time = datetime.now()
total = (end_time - start_time).total_seconds() / 60

print(f"\n⏰ Tiempo total: {total:.1f} minutos ({total/60:.1f} horas)")
print(f"💾 Modelos en: {analyzer.output_dir}/medium/")

## 7️⃣ Descargar modelos

In [ ]:
# Comprimir modelos
!cd models/temporal && zip -r customer_v3_medium.zip customer_v3/medium/

print("✅ Modelos comprimidos")
print("\n📥 Descargando...")

from google.colab import files
files.download('models/temporal/customer_v3_medium.zip')

print("\n✅ Descarga iniciada")
print("Descomprime en: E:\\Codigos\\Proyecto Final\\models\\temporal\\")

## 8️⃣ Ver métricas

In [ ]:
!cat models/temporal/customer_v3/medium/metrics.json